# CIFAR-10N noisy labels: Inter-IIT CV bootcamp (complete pipeline)

Self-contained: every source file is written by the cells below. Everything (data, raw outputs, predictions,
the final zip) lives in Google Drive, so **a disconnect loses nothing**. Re-run the notebook from the top and
finished runs are skipped automatically.

**Before you start**
1. `Runtime → Change runtime type → GPU` (T4 is fine; L4/A100 is faster).
2. In Google Drive, create the folder `MyDrive/cv_bootcamp/data/` and upload the 6 provided data files into it:
   `classes.txt, labels.csv, test_images.npy, train_idx.txt, train_images.npy, val_idx.txt`.
3. Fill in `NAME` / `ROLL_NO` in the next cell, then `Runtime → Run all`.

Approximate time: T4 ~3.5–4 h, L4 ~2 h, A100 ~1 h.

In [ ]:
NAME = "YOUR NAME"        # <-- fill in
ROLL_NO = "24B1534"       # <-- fill in
ROOT = "/content/drive/MyDrive/cv_bootcamp"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{ROOT}/submission/code", exist_ok=True)
os.makedirs(f"{ROOT}/submission/predictions", exist_ok=True)
os.makedirs(f"{ROOT}/work", exist_ok=True)
need = ["classes.txt", "labels.csv", "test_images.npy", "train_idx.txt", "train_images.npy", "val_idx.txt"]
missing = [f for f in need if not os.path.exists(f"{ROOT}/data/{f}")]
assert not missing, f"Upload these to {ROOT}/data/: {missing}"
print("data OK")

## Source files (written to Drive)

In [ ]:
%%writefile /content/drive/MyDrive/cv_bootcamp/submission/code/model.py
"""ResNet-18 for 32x32 CIFAR images (the standard CIFAR variant: 3x3 stem, no max-pool).

Everyone must use this exact architecture, trained from scratch (no pretrained weights).

    from model import ResNet18
    net = ResNet18(num_classes=10)

You may wrap it (e.g. two copies for co-training) but do not change the layers.
"""
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNet(nn.Module):
    def __init__(self, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, planes, n, stride):
        layers = []
        for s in [stride] + [1] * (n - 1):
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.linear(out)


def ResNet18(num_classes=10):
    return ResNet([2, 2, 2, 2], num_classes)


if __name__ == "__main__":
    import torch
    net = ResNet18()
    print("parameters:", sum(p.numel() for p in net.parameters()))  # ~11.17M
    print("output shape:", net(torch.randn(2, 3, 32, 32)).shape)    # (2, 10)

In [ ]:
%%writefile /content/drive/MyDrive/cv_bootcamp/submission/code/nl.py
"""Learning from noisy labels on CIFAR-10N (Inter-IIT CV bootcamp). Every task uses this file.

    python nl.py train --tag t1_noisy --labels noisy --method ce --track
    python nl.py cv --labels noisy
    python nl.py summarize

See README.md for the full sequence. Raw outputs (per-epoch logits, histories) go to WORK_DIR;
`summarize` turns them into predictions/, results.json and stats used in report.md.
"""
import argparse
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from model import ResNet18  # noqa: E402

SUB = HERE.parent
# data/ and work/ inside the repo if present (a full clone), otherwise next to the submission folder
DATA = Path(os.environ.get("DATA_DIR", SUB / "data" if (SUB / "data").is_dir() else HERE.parents[1] / "data"))
WORK = Path(os.environ.get("WORK_DIR", SUB / "work" if (SUB / "work").is_dir() else HERE.parents[1] / "work"))
PRED = SUB / "predictions"

# Shared training setup: identical for the baseline and every method.
SEED = 0
EPOCHS = 60
BATCH = 128
LR = 0.1
WD = 5e-4
MOMENTUM = 0.9
WARMUP_EPOCHS = 1
CV_FOLDS = 4
CV_EPOCHS = 20
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
DEV = "cuda"
# bf16 on Ampere and newer; fp16 with loss scaling on older GPUs (e.g. Colab's T4)
AMP = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
COLS = {"noisy": 1, "synthetic": 2, "clean": 3}


# ---------------------------------------------------------------- data
def load_data():
    lab = np.loadtxt(DATA / "labels.csv", delimiter=",", skiprows=1, dtype=np.int64)
    assert (lab[:, 0] == np.arange(len(lab))).all()
    d = {k: lab[:, c] for k, c in COLS.items()}
    d["train_idx"] = np.loadtxt(DATA / "train_idx.txt", dtype=np.int64)
    d["val_idx"] = np.loadtxt(DATA / "val_idx.txt", dtype=np.int64)
    return d


def to_gpu(images):
    x = torch.from_numpy(images).to(DEV).permute(0, 3, 1, 2).float().div_(255)
    return ((x - MEAN.to(DEV)) / STD.to(DEV)).half().contiguous()


def augment(x):
    """Random 32x32 crop from the zero-padded (4px) image plus horizontal flip, done on the GPU."""
    b = x.shape[0]
    x = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (b,), device=DEV)
    j = torch.randint(0, 9, (b,), device=DEV)
    ar = torch.arange(32, device=DEV)
    rows = (i[:, None] + ar)[:, None, :, None]
    cols = (j[:, None] + ar)[:, None, None, :]
    x = x[torch.arange(b, device=DEV)[:, None, None, None], torch.arange(3, device=DEV)[None, :, None, None],
          rows, cols]
    flip = torch.rand(b, device=DEV) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)


# ---------------------------------------------------------------- losses
class CE:
    def __call__(self, out, y, idx):
        return F.cross_entropy(out, y)


class GCE:
    """Generalized cross-entropy (Zhang & Sabuncu 2018): (1 - p_y^q) / q."""

    def __init__(self, q=0.7):
        self.q = q

    def __call__(self, out, y, idx):
        p = F.softmax(out.float(), 1).gather(1, y[:, None]).squeeze(1).clamp_min(1e-7)
        return ((1 - p ** self.q) / self.q).mean()


class ELR:
    """Early-learning regularization (Liu et al. 2020): CE + lam * log(1 - <t_i, p_i>),
    where t_i is a running average of the model's own predictions for sample i."""

    def __init__(self, n, lam=3.0, beta=0.7):
        self.t = torch.zeros(n, 10, device=DEV)
        self.lam, self.beta = lam, beta

    def __call__(self, out, y, idx):
        p = F.softmax(out.float(), 1).clamp(1e-4, 1 - 1e-4)
        self.t[idx] = self.beta * self.t[idx] + (1 - self.beta) * (p / p.sum(1, keepdim=True)).detach()
        reg = torch.log(1 - (self.t[idx] * p).sum(1)).mean()
        return F.cross_entropy(out, y) + self.lam * reg


# ---------------------------------------------------------------- training
@torch.no_grad()
def predict(model, x, bs=1000):
    model.eval()
    outs = []
    for k in range(0, len(x), bs):
        with torch.autocast("cuda", dtype=AMP):
            outs.append(model(x[k:k + bs].contiguous(memory_format=torch.channels_last)).float())
    return torch.cat(outs)


def fit(x_train, y_train, loss_fn, eval_sets, epochs=EPOCHS, seed=SEED, log=print):
    """Train a fresh ResNet-18. Returns {name: (epochs, n, 10) float16 logits} for every eval set,
    evaluated after each epoch, plus the per-epoch mean training loss."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(x_train)
    steps_per_epoch = n // BATCH
    total = epochs * steps_per_epoch
    warm = WARMUP_EPOCHS * steps_per_epoch
    model = ResNet18(10).to(DEV, memory_format=torch.channels_last)
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, nesterov=True, weight_decay=WD)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP == torch.float16)
    logits = {k: [] for k in eval_sets}
    losses = []
    for ep in range(epochs):
        t0 = time.time()
        model.train()
        perm = torch.randperm(n, device=DEV)
        run = torch.zeros((), device=DEV)
        for s in range(steps_per_epoch):
            idx = perm[s * BATCH:(s + 1) * BATCH]
            xb = augment(x_train[idx]).contiguous(memory_format=torch.channels_last)
            with torch.autocast("cuda", dtype=AMP):
                out = model(xb)
            loss = loss_fn(out.float(), y_train[idx], idx)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run += loss.detach()
        losses.append(run.item() / steps_per_epoch)
        msg = []
        for k, (xe, ye) in eval_sets.items():
            out = predict(model, xe)
            logits[k].append(out.half().cpu().numpy())
            if ye is not None:
                msg.append(f"{k} {(out.argmax(1) == ye).float().mean().item():.4f}")
        log(f"ep {ep + 1:2d}/{epochs} loss {losses[-1]:.4f} {' '.join(msg)} ({time.time() - t0:.1f}s)")
    return {k: np.stack(v) for k, v in logits.items()}, losses


def make_logger(path):
    f = open(path, "a", encoding="utf-8")

    def log(m):
        print(m, flush=True)
        f.write(m + "\n")
        f.flush()
    return log


# ---------------------------------------------------------------- noise scores (Task 3)
def cv_probs(labels):
    """Held-out softmax for every training image from a CV_FOLDS-fold split (mean of the last 5 epochs)."""
    p = np.load(WORK / f"cv_{labels}" / "heldout_probs.npy")
    return p


def noise_scores(d, labels, ce_tag):
    """Final Task 3 score, fixed in advance: mean of the rank-normalised
    (a) cross-validated probability that the given label is wrong, 1 - p_cv[y], and
    (b) per-sample cross-entropy of the given label averaged over all epochs of the CE run on the same labels."""
    tr = d["train_idx"]
    y = d[labels][tr]
    p = cv_probs(labels)
    s_cv = 1 - p[np.arange(len(tr)), y]
    s_loss = mean_train_loss(ce_tag, y)
    return 0.5 * (rank01(s_cv) + rank01(s_loss)), s_cv, s_loss


def mean_train_loss(tag, y):
    lg = np.load(WORK / tag / "train_logits.npy").astype(np.float32)  # (E, N, 10)
    lsm = lg - np.log(np.exp(lg - lg.max(2, keepdims=True)).sum(2, keepdims=True)) - lg.max(2, keepdims=True)
    return -lsm[:, np.arange(len(y)), y].mean(0)


def rank01(s):
    r = np.empty(len(s))
    r[np.argsort(s, kind="stable")] = np.arange(len(s))
    return r / (len(s) - 1)


def relabeled_set(d, labels, ce_tag, conf_thr=0.5):
    """Flag the top-r scoring images, where r is the CV disagreement rate (an estimate of the noise rate
    from noisy labels only). Flagged images whose CV prediction is confident are relabelled to it; the rest
    are dropped. Returns (positions into train_idx to keep, their new labels, info)."""
    tr = d["train_idx"]
    y = d[labels][tr].copy()
    score, _, _ = noise_scores(d, labels, ce_tag)
    p = cv_probs(labels)
    pred, conf = p.argmax(1), p.max(1)
    rate = float((pred != y).mean())
    flagged = np.argsort(-score)[:int(round(rate * len(tr)))]
    relabel = flagged[(conf[flagged] >= conf_thr) & (pred[flagged] != y[flagged])]
    drop = np.setdiff1d(flagged, relabel)
    y[relabel] = pred[relabel]
    keep = np.setdiff1d(np.arange(len(tr)), drop)
    return keep, y[keep], {"est_noise_rate": rate, "flagged": len(flagged), "relabelled": len(relabel),
                           "dropped": len(drop)}


# ---------------------------------------------------------------- commands
def cmd_train(a):
    d = load_data()
    out = WORK / a.tag
    out.mkdir(parents=True, exist_ok=True)
    log = make_logger(out / "log.txt")
    imgs = to_gpu(np.load(DATA / "train_images.npy"))
    test = to_gpu(np.load(DATA / "test_images.npy"))
    tr, va = d["train_idx"], d["val_idx"]
    pos = np.arange(len(tr))
    y_np = d[a.labels][tr]
    info = {}
    if a.method == "relabel":
        pos, y_np, info = relabeled_set(d, a.labels, a.ce_tag)
        log(f"relabel: {info}")
    x_train = imgs[torch.from_numpy(tr[pos]).to(DEV)]
    y_train = torch.from_numpy(y_np).to(DEV)
    loss_fn = {"ce": lambda: CE(), "relabel": lambda: CE(), "gce": lambda: GCE(),
               "elr": lambda: ELR(len(pos))}[a.method]()
    sel = "synthetic" if a.labels == "synthetic" else "noisy"
    eval_sets = {f"val_{sel}": (imgs[torch.from_numpy(va).to(DEV)], torch.from_numpy(d[sel][va]).to(DEV)),
                 "test": (test, None)}
    if a.track:
        eval_sets["train"] = (imgs[torch.from_numpy(tr).to(DEV)], None)
    log(f"== {a.tag}: method={a.method} labels={a.labels} n_train={len(pos)} epochs={EPOCHS}")
    logits, losses = fit(x_train, y_train, loss_fn, eval_sets, log=log)
    np.save(out / "val_logits.npy", logits[f"val_{sel}"])
    np.save(out / "test_logits.npy", logits["test"])
    if a.track:
        np.save(out / "train_logits.npy", logits["train"])
    json.dump({"method": a.method, "labels": a.labels, "train_loss": losses, "info": info},
              open(out / "meta.json", "w"), indent=1)


def cmd_cv(a):
    d = load_data()
    out = WORK / f"cv_{a.labels}"
    out.mkdir(parents=True, exist_ok=True)
    log = make_logger(out / "log.txt")
    imgs = to_gpu(np.load(DATA / "train_images.npy"))
    tr = d["train_idx"]
    y = d[a.labels][tr]
    rng = np.random.RandomState(SEED)
    fold = rng.permutation(len(tr)) % CV_FOLDS
    probs = np.zeros((len(tr), 10), np.float32)
    for k in range(CV_FOLDS):
        fit_pos, held = np.where(fold != k)[0], np.where(fold == k)[0]
        log(f"== cv_{a.labels} fold {k + 1}/{CV_FOLDS}: train {len(fit_pos)} held-out {len(held)}")
        x_fit = imgs[torch.from_numpy(tr[fit_pos]).to(DEV)]
        x_held = imgs[torch.from_numpy(tr[held]).to(DEV)]
        logits, _ = fit(x_fit, torch.from_numpy(y[fit_pos]).to(DEV), CE(),
                        {"heldout": (x_held, torch.from_numpy(y[held]).to(DEV))}, epochs=CV_EPOCHS,
                        seed=SEED + k, log=log)
        lg = torch.from_numpy(logits["heldout"][-5:].astype(np.float32))
        probs[held] = F.softmax(lg, 2).mean(0).numpy()
    np.save(out / "heldout_probs.npy", probs)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sp = ap.add_subparsers(dest="cmd", required=True)
    t = sp.add_parser("train")
    t.add_argument("--tag", required=True)
    t.add_argument("--labels", choices=["noisy", "synthetic", "clean"], required=True)
    t.add_argument("--method", choices=["ce", "gce", "elr", "relabel"], default="ce")
    t.add_argument("--track", action="store_true", help="also log logits on the whole training set every epoch")
    t.add_argument("--ce-tag", help="CE run (with --track) on the same labels; used by --method relabel")
    c = sp.add_parser("cv")
    c.add_argument("--labels", choices=["noisy", "synthetic"], required=True)
    sp.add_parser("summarize")
    args = ap.parse_args()
    torch.backends.cudnn.benchmark = True
    if args.cmd == "summarize":
        from summarize import main as summarize_main
        summarize_main()
    else:
        {"train": cmd_train, "cv": cmd_cv}[args.cmd](args)

In [ ]:
%%writefile /content/drive/MyDrive/cv_bootcamp/submission/code/summarize.py
"""Turn the raw run outputs in WORK_DIR into predictions/, results.json and code/analysis.md.

Clean labels are used here only to *measure* (confusion matrix, Task 2 curves, Task 3 AUROC, and
clean-label validation accuracy reported for analysis). Every selection uses noisy (Tasks 1-4) or
synthetic (Task 5) validation labels.
"""
import csv
import json

import numpy as np

from nl import BATCH, EPOCHS, PRED, SUB, WORK, load_data, noise_scores

T4_METHODS = ["elr", "relabel", "gce"]
T4_NAMES = {"elr": "ELR (early-learning regularization)",
            "relabel": "CV-score filter + relabel, then CE retrain",
            "gce": "GCE (generalized cross-entropy, q=0.7)"}
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
_D = None


def data():
    global _D
    if _D is None:
        _D = load_data()
    return _D


def has(tag):
    return (WORK / tag / "val_logits.npy").exists()


def val_curve(tag, labels):
    """Per-epoch validation accuracy of run `tag` against the given label column."""
    d = data()
    lg = np.load(WORK / tag / "val_logits.npy")
    return (lg.argmax(2) == d[labels][d["val_idx"]]).mean(1)


def best_t4_method():
    done = [m for m in T4_METHODS if has(f"t4_{m}")]
    return max(done, key=lambda m: val_curve(f"t4_{m}", "noisy").max())


def test_preds(tag, epoch):
    """Test predictions after `epoch` (1-based)."""
    return np.load(WORK / tag / "test_logits.npy")[epoch - 1].astype(np.float32).argmax(1)


def write_preds(name, pred):
    with open(PRED / name, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "pred"])
        w.writerows(zip(range(len(pred)), pred.tolist()))


def auroc(score, pos):
    r = np.empty(len(score))
    order = np.argsort(score, kind="stable")
    r[order] = np.arange(1, len(score) + 1)
    # average ranks over ties
    s_sorted = score[order]
    _, start, counts = np.unique(s_sorted, return_index=True, return_counts=True)
    for st, c in zip(start, counts):
        if c > 1:
            r[order[st:st + c]] = st + (c + 1) / 2
    n1, n0 = pos.sum(), (~pos).sum()
    return float((r[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def prec_at_k(score, pos):
    k = int(pos.sum())
    return float(pos[np.argsort(-score)[:k]].mean())


def memorization_start(acc_m, margin=0.03):
    """First epoch (1-based) from which the 3-epoch moving average of the accuracy on the *wrong* labels of
    the mislabeled subset stays more than `margin` above its early plateau (median of epochs 5-20)."""
    s = np.convolve(acc_m, np.ones(3) / 3, "same")
    s[0], s[-1] = acc_m[:2].mean(), acc_m[-2:].mean()
    above = s > np.median(acc_m[4:20]) + margin
    return next(i for i in range(len(s)) if above[i:].all()) + 1


def r4(x):
    return None if x is None else round(float(x), 4)


def main():
    d = data()
    tr, va = d["train_idx"], d["val_idx"]
    PRED.mkdir(exist_ok=True)
    res = json.load(open(SUB / "results.json")) if (SUB / "results.json").exists() else {}
    res.setdefault("name", "YOUR NAME")
    res.setdefault("roll_no", "YOUR_ROLL_NO")
    res["seed"] = 0
    res["training_setup"] = {
        "optimizer": "SGD, momentum 0.9 (Nesterov), weight decay 5e-4, bf16 autocast",
        "learning_rate_schedule": "linear warmup to 0.1 over 1 epoch, then per-step cosine decay to 0",
        "batch_size": BATCH,
        "epochs": EPOCHS,
        "augmentation": "random 32x32 crop from 4px zero-padded image + random horizontal flip",
        "gpu_used": "NVIDIA GeForce RTX 4050 Laptop GPU (6 GB)",
    }
    A = []  # analysis.md lines
    stats = {}

    # ---- Task 1
    noisy, clean = d["noisy"][tr], d["clean"][tr]
    wrong = noisy != clean
    rate = float(wrong.mean())
    cm = np.zeros((10, 10), int)
    np.add.at(cm, (clean, noisy), 1)
    off = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j]
    off.sort(reverse=True)
    pair = {}
    for c, a, b in off:
        pair[tuple(sorted((a, b)))] = pair.get(tuple(sorted((a, b))), 0) + c
    pairs = sorted(pair.items(), key=lambda kv: -kv[1])
    stats["noise_rate_train"] = rate
    stats["noise_rate_all50k"] = float((d["noisy"] != d["clean"]).mean())
    stats["top_directed"] = [(int(c), a, b, round(c / wrong.sum(), 4)) for c, a, b in off[:10]]
    stats["top_pairs"] = [(a, b, int(c), round(c / wrong.sum(), 4)) for (a, b), c in pairs[:8]]
    A += ["# Analysis (supporting numbers for report.md)", "",
          f"## Task 1: noise rate {rate:.4f} on the 45k training split "
          f"({stats['noise_rate_all50k']:.4f} on all 50k)", "",
          "Confusion matrix, rows = clean label, columns = noisy label (training split):", "",
          "| clean \\ noisy | " + " | ".join(CLASSES) + " |", "|---" * 11 + "|"]
    for i in range(10):
        A.append(f"| {CLASSES[i]} | " + " | ".join(str(v) for v in cm[i]) + " |")
    A += ["", "Per-class noise rate (fraction of each clean class that got a wrong label): " +
          ", ".join(f"{CLASSES[i]} {1 - cm[i, i] / cm[i].sum():.3f}" for i in range(10)), "",
          "Largest undirected confusion pairs (share of all wrong labels): " +
          "; ".join(f"{a}-{b} {c} ({s:.1%})" for a, b, c, s in stats["top_pairs"]), ""]
    t1 = {"noise_rate": r4(rate)}
    for tag, key, fname in [("t1_clean", "val_acc_noisy_labels__clean_ref_model", "t1_clean_ref.csv"),
                            ("t1_noisy", "val_acc_noisy_labels__noisy_ce_model", "t1_noisy_ce.csv")]:
        if has(tag):
            write_preds(fname, test_preds(tag, EPOCHS))
            t1[key] = r4(val_curve(tag, "noisy")[-1])
            stats[tag] = {"val_noisy_final": t1[key], "val_clean_final": r4(val_curve(tag, "clean")[-1])}
    res["task1"] = {**res.get("task1", {}), **t1}
    A += ["Final-epoch validation accuracy: " + json.dumps({k: v for k, v in stats.items() if k.startswith("t1_")}),
          ""]

    # ---- Task 2
    if (WORK / "t1_noisy" / "train_logits.npy").exists():
        tp = np.load(WORK / "t1_noisy" / "train_logits.npy").argmax(2)  # (E, N)
        acc_c = (tp[:, ~wrong] == noisy[~wrong]).mean(1)
        acc_m = (tp[:, wrong] == noisy[wrong]).mean(1)
        acc_m_true = (tp[:, wrong] == clean[wrong]).mean(1)
        vn, vc = val_curve("t1_noisy", "noisy"), val_curve("t1_noisy", "clean")
        with open(PRED / "t2_dynamics.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"])
            for e in range(len(vn)):
                w.writerow([e + 1, f"{acc_c[e]:.4f}", f"{acc_m[e]:.4f}", f"{vn[e]:.4f}", f"{vc[e]:.4f}"])
        mem = memorization_start(acc_m)
        es = int(np.argmax(vn)) + 1
        write_preds("t2_early_stop.csv", test_preds("t1_noisy", es))
        res["task2"] = {"memorization_start_epoch": mem, "early_stop_epoch": es,
                        "val_acc_noisy_labels_at_early_stop": r4(vn[es - 1])}
        stats["t2"] = {"mem_epoch": mem, "acc_mis_at_mem": r4(acc_m[mem - 1]), "acc_mis_final": r4(acc_m[-1]),
                       "acc_mis_true_at_mem": r4(acc_m_true[mem - 1]), "acc_mis_true_final": r4(acc_m_true[-1]),
                       "acc_correct_at_mem": r4(acc_c[mem - 1]), "acc_correct_final": r4(acc_c[-1]),
                       "val_clean_peak_epoch": int(np.argmax(vc)) + 1, "val_clean_peak": r4(vc.max()),
                       "val_clean_final": r4(vc[-1]), "early_stop": es, "val_noisy_es": r4(vn[es - 1]),
                       "val_clean_es": r4(vc[es - 1]), "val_noisy_final": r4(vn[-1])}
        A += ["## Task 2", "", "| epoch | train_acc_correct | train_acc_mislabeled (fits wrong label) | "
              "mislabeled predicted as clean label | val_noisy | val_clean |", "|---" * 6 + "|"]
        A += [f"| {e + 1} | {acc_c[e]:.4f} | {acc_m[e]:.4f} | {acc_m_true[e]:.4f} | {vn[e]:.4f} | {vc[e]:.4f} |"
              for e in range(len(vn))]
        A += ["", json.dumps(stats["t2"]), ""]

    # ---- Task 3
    if (WORK / "cv_noisy" / "heldout_probs.npy").exists() and (WORK / "t1_noisy" / "train_logits.npy").exists():
        score, s_cv, s_loss = noise_scores(d, "noisy", "t1_noisy")
        with open(PRED / "t3_noise_scores.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["index", "score"])
            w.writerows((int(i), f"{s:.6f}") for i, s in zip(tr, score))
        res["task3"] = {"method": "mean of rank-normalised (a) 1 - cross-validated softmax prob of the noisy label "
                                  "(4-fold CV, 20 epochs per fold, mean of last 5 epochs) and (b) per-sample CE loss "
                                  "averaged over all 60 epochs of the Task 1 noisy CE run"}
        stats["t3"] = {}
        for name, s in [("combined (submitted)", score), ("cv_confidence", s_cv), ("mean_train_loss", s_loss)]:
            stats["t3"][name] = {"auroc": r4(auroc(s, wrong)), "prec_at_k": r4(prec_at_k(s, wrong))}
        lg = np.load(WORK / "t1_noisy" / "train_logits.npy")
        for e in [stats.get("t2", {}).get("early_stop"), EPOCHS]:
            if e:
                l = lg[e - 1].astype(np.float32)
                l = l - l.max(1, keepdims=True)
                loss_e = -(l[np.arange(len(tr)), noisy] - np.log(np.exp(l).sum(1)))
                stats["t3"][f"loss_at_epoch_{e}"] = {"auroc": r4(auroc(loss_e, wrong)),
                                                     "prec_at_k": r4(prec_at_k(loss_e, wrong))}
        # failure mode: where the score misses (mislabeled images ranked low) by class pair
        k = int(wrong.sum())
        flagged = np.zeros(len(tr), bool)
        flagged[np.argsort(-score)[:k]] = True
        miss = wrong & ~flagged
        recall_pair = {}
        for (a, b), _ in pairs[:6]:
            ia, ib = CLASSES.index(a), CLASSES.index(b)
            m = wrong & (((clean == ia) & (noisy == ib)) | ((clean == ib) & (noisy == ia)))
            recall_pair[f"{a}-{b}"] = r4(flagged[m].mean())
        stats["t3"]["recall_at_k_overall"] = r4(flagged[wrong].mean())
        stats["t3"]["recall_at_k_by_pair"] = recall_pair
        stats["t3"]["missed_mislabeled"] = int(miss.sum())
        A += ["## Task 3 (clean labels used only to measure)", "", json.dumps(stats["t3"], indent=1), ""]

    # ---- Task 4
    methods = []
    rows = []
    if "t2" in stats:
        rows.append(("CE baseline (early-stopped, Task 2)", stats["t2"]["val_noisy_es"], stats["t2"]["val_clean_es"],
                     stats["t1_noisy"]["val_noisy_final"], stats["t1_noisy"]["val_clean_final"]))
    for m in T4_METHODS:
        tag = f"t4_{m}"
        if not has(tag):
            continue
        vn, vc = val_curve(tag, "noisy"), val_curve(tag, "clean")
        e = int(np.argmax(vn)) + 1
        write_preds(f"t4_{m}.csv", test_preds(tag, e))
        methods.append({"name": m, "file": f"t4_{m}.csv", "val_acc_noisy_labels": r4(vn[e - 1])})
        rows.append((T4_NAMES[m] + f" [best epoch {e}]", r4(vn[e - 1]), r4(vc[e - 1]), r4(vn[-1]), r4(vc[-1])))
        meta = json.load(open(WORK / tag / "meta.json"))
        if meta.get("info"):
            stats["relabel_info"] = meta["info"]
    if methods:
        best = max(methods, key=lambda x: x["val_acc_noisy_labels"])["name"]
        res["task4"] = {"methods": methods, "best_method": best}
        stats["t4_best"] = best
    stats["t4_rows"] = rows
    A += ["## Task 4 (noisy labels; epoch picked by noisy val acc)", "",
          "| method | val noisy @sel | val clean @sel | val noisy final | val clean final |", "|---" * 5 + "|"]
    A += [f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} |" for r in rows]
    if "relabel_info" in stats:
        A += ["", "Relabel step: " + json.dumps(stats["relabel_info"])]
    A.append("")

    # ---- Task 5
    t5 = {}
    rows5 = []
    for tag, key, fname in [("t5_ce", "val_acc_synthetic_labels__baseline", "t5_baseline_synth.csv"),
                            ("t5_best", "val_acc_synthetic_labels__best_method", "t5_best_synth.csv")]:
        if has(tag):
            vs, vc = val_curve(tag, "synthetic"), val_curve(tag, "clean")
            e = int(np.argmax(vs)) + 1
            write_preds(fname, test_preds(tag, e))
            t5[key] = r4(vs[e - 1])
            rows5.append((tag + f" [best epoch {e}]", r4(vs[e - 1]), r4(vc[e - 1]), r4(vs[-1]), r4(vc[-1])))
    if (WORK / "t5_ce" / "train_logits.npy").exists():
        syn_wrong = d["synthetic"][tr] != clean
        tp = np.load(WORK / "t5_ce" / "train_logits.npy").argmax(2)
        acc_m5 = (tp[:, syn_wrong] == d["synthetic"][tr][syn_wrong]).mean(1)
        stats["t5_synth_noise_rate"] = r4(syn_wrong.mean())
        stats["t5_mem_epoch"] = memorization_start(acc_m5)
        stats["t5_acc_mis_final"] = r4(acc_m5[-1])
        if (WORK / "cv_synthetic" / "heldout_probs.npy").exists():
            sc, scv, sl = noise_scores(d, "synthetic", "t5_ce")
            stats["t5_noise_score_auroc"] = r4(auroc(sc, syn_wrong))
    res["task5"] = {**res.get("task5", {}), **t5}
    stats["t5_rows"] = rows5
    A += ["## Task 5 (synthetic labels; epoch picked by synthetic val acc)", "",
          "| run | val synth @sel | val clean @sel | val synth final | val clean final |", "|---" * 5 + "|"]
    A += [f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} |" for r in rows5]
    A += ["", json.dumps({k: v for k, v in stats.items() if k.startswith("t5_") and "rows" not in k}), ""]

    json.dump(res, open(SUB / "results.json", "w"), indent=2)
    json.dump(stats, open(WORK / "stats.json", "w"), indent=1)
    (SUB / "code" / "analysis.md").write_text("\n".join(A), encoding="utf-8")
    print(json.dumps(res, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/cv_bootcamp/submission/code/README.md
# CIFAR-10N: learning from noisy labels

All code is in `nl.py` (data, GPU augmentation, training loop, losses, Task 3 scores) and
`summarize.py` (writes `predictions/`, `results.json` and `analysis.md`). `model.py` is the provided ResNet-18, unchanged.

## Setup

- Python 3.13, PyTorch 2.13 (CUDA 13.0), NumPy. One NVIDIA GPU (tested on an RTX 4050 Laptop, 6 GB).
- Data: the provided `data/` folder. The code uses `data/` at the submission root if it exists (as in the GitHub
  repo), otherwise `data/` one level above the submission folder; override with `DATA_DIR=/path/to/data`.
- Raw run outputs (per-epoch logits) go to `work/`, located the same way (override with `WORK_DIR`).

## Shared training setup (identical for every run)

ResNet-18 from scratch, SGD (momentum 0.9, Nesterov, weight decay 5e-4), batch 128, 60 epochs,
linear warm-up to lr 0.1 over 1 epoch then per-step cosine decay to 0, bf16 autocast,
random crop (4 px zero padding) + horizontal flip, seed 0. After every epoch the model is evaluated on the
validation and test sets, and the logits are saved, so any epoch can be selected afterwards.

## Reproduce everything

```bash
bash run_all.sh        # about 3 h on an RTX 4050 laptop GPU
```

or step by step:

| Task | Command | Output |
|---|---|---|
| 1 | `python nl.py train --tag t1_noisy --labels noisy --method ce --track` | noisy CE baseline; `--track` also logs training-set logits per epoch (Task 2, Task 3) |
| 1 | `python nl.py train --tag t1_clean --labels clean --method ce` | clean-label reference |
| 2 | (from `t1_noisy`) | `t2_dynamics.csv`; early-stop epoch = argmax of noisy val accuracy |
| 3 | `python nl.py cv --labels noisy` | 4-fold CV (4 runs, 20 epochs each), held-out softmax for all 45k images |
| 4 | `python nl.py train --tag t4_elr --labels noisy --method elr` | Early-Learning Regularization (λ=3, β=0.7) |
| 4 | `python nl.py train --tag t4_relabel --labels noisy --method relabel --ce-tag t1_noisy` | filter/relabel with Task 3 scores, then CE retrain |
| 4 | `python nl.py train --tag t4_gce --labels noisy --method gce` | Generalized CE (q=0.7) |
| 5 | `python nl.py train --tag t5_ce --labels synthetic --method ce --track` | CE baseline on synthetic labels |
| 5 | `python nl.py train --tag t5_best --labels synthetic --method <best>` | best Task 4 method on synthetic labels (for `relabel`, first run `python nl.py cv --labels synthetic` and pass `--ce-tag t5_ce`) |
| all | `python nl.py summarize` | `predictions/*.csv`, `results.json`, `analysis.md` |

## Method details

- **Model selection.** Task 1 predictions come from the final (60th) epoch. Task 2, Task 4 and the Task 5 runs use the epoch
  with the highest validation accuracy, measured with noisy labels for Tasks 2 and 4 and with synthetic labels for Task 5.
- **Task 3 score**, fixed before looking at any clean-label numbers: the mean of the rank-normalised
  (a) `1 - p_cv(noisy label)` from 4-fold cross-validation (mean softmax over the last 5 of 20 epochs) and
  (b) the per-sample CE of the noisy label averaged over all 60 epochs of the Task 1 noisy run.
- **Relabel method.** Flag the top-r images by score, where r is the CV disagreement rate (an estimate of the noise rate
  that uses only noisy labels). A flagged image whose CV prediction has confidence ≥ 0.5 is relabelled to that
  prediction; the other flagged images are dropped. Then train CE on the result with the shared setup.
- **Clean labels** are used only to train `t1_clean` and to *measure* things in `summarize.py`:
  the confusion matrix, the Task 2 curves, Task 3 AUROC / precision@k, and clean validation accuracy reported in
  `analysis.md`. They never select, filter or relabel anything.

In [ ]:
%%writefile /content/drive/MyDrive/cv_bootcamp/check_format.py
"""Check that your CV submission has the right files and formats BEFORE you submit.

Usage (run from this folder, so it can find data/):
    python check_format.py path/to/<rollno>_cv_bootcamp.zip
    python check_format.py path/to/your_submission_folder

It only checks names and formats. It does NOT score your predictions.
We use the same checks when grading, so a file that fails here gets 0 marks.
"""
import csv
import json
import math
import os
import re
import shutil
import sys
import tempfile
import zipfile

HERE = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(HERE, "data")

N_TEST = 10_000
N_TRAIN = 45_000
MAX_EPOCHS = 60
NUM_CLASSES = 10

# id,pred files with a fixed name
PRED_FILES = ["t1_clean_ref.csv", "t1_noisy_ce.csv", "t2_early_stop.csv",
              "t5_baseline_synth.csv", "t5_best_synth.csv"]
DYNAMICS_FILE = "t2_dynamics.csv"
DYNAMICS_HEADER = ["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"]
SCORES_FILE = "t3_noise_scores.csv"
T4_PATTERN = re.compile(r"^t4_.+\.csv$")
MIN_T4, MAX_T4 = 1, 3

TASKS = {
    "task1": ["t1_clean_ref.csv", "t1_noisy_ce.csv"],
    "task2": [DYNAMICS_FILE, "t2_early_stop.csv"],
    "task3": [SCORES_FILE],
    "task4": None,  # 1-3 files matching t4_<method>.csv, handled separately
    "task5": ["t5_baseline_synth.csv", "t5_best_synth.csv"],
}


# ---------------------------------------------------------------- helpers
def locate_root(path):
    """Return (root_folder, temp_dir_or_None). root_folder contains predictions/."""
    tmp = None
    if os.path.isfile(path) and path.lower().endswith(".zip"):
        tmp = tempfile.mkdtemp()
        with zipfile.ZipFile(path) as z:
            z.extractall(tmp)
        path = tmp
    for dirpath, dirnames, _ in os.walk(path):
        if "__MACOSX" in dirpath:
            continue
        if "predictions" in dirnames:
            return dirpath, tmp
    return None, tmp


def read_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        rows = [[c.strip() for c in r] for r in csv.reader(f) if any(c.strip() for c in r)]
    if not rows:
        raise ValueError("file is empty")
    return [h.lower() for h in rows[0]], rows[1:]


def to_int(s, what):
    if not re.fullmatch(r"[+-]?\d+", s):
        raise ValueError(f"{what} must be an integer, got '{s}'")
    return int(s)


def to_float(s, what):
    try:
        v = float(s)
    except ValueError:
        raise ValueError(f"{what} must be a number, got '{s}'") from None
    if not math.isfinite(v):
        raise ValueError(f"{what} must be finite, got '{s}'")
    return v


def load_train_idx(data_dir=DATA_DIR):
    path = os.path.join(data_dir, "train_idx.txt")
    with open(path, encoding="utf-8") as f:
        return {int(line) for line in f if line.strip()}


# ---------------------------------------------------------------- per-file checks
def check_pred_file(path, warnings=None):
    """id,pred with exactly 10,000 rows, ids 0..9999 once each, pred in 0..9. Returns {id: pred}."""
    header, rows = read_csv(path)
    if header != ["id", "pred"]:
        raise ValueError("header must be 'id,pred'")
    preds = {}
    for i, r in enumerate(rows, 2):
        if len(r) != 2:
            raise ValueError(f"row {i}: need exactly 2 columns")
        pid = to_int(r[0], f"row {i}: id")
        pred = to_int(r[1], f"row {i}: pred")
        if not 0 <= pid < N_TEST:
            raise ValueError(f"row {i}: id {pid} outside 0..{N_TEST - 1}")
        if not 0 <= pred < NUM_CLASSES:
            raise ValueError(f"row {i}: pred {pred} outside 0..{NUM_CLASSES - 1}")
        if pid in preds:
            raise ValueError(f"row {i}: id {pid} appears twice")
        preds[pid] = pred
    if len(preds) != N_TEST:
        missing = sorted(set(range(N_TEST)) - set(preds))
        raise ValueError(f"need {N_TEST} rows (ids 0..{N_TEST - 1}), found {len(preds)}; "
                         f"missing e.g. {missing[:3]}")
    if warnings is not None and len(set(preds.values())) == 1:
        warnings.append(f"{os.path.basename(path)}: every prediction is the same class")
    return preds


def check_dynamics(path, warnings=None):
    """Per-epoch log. Returns list of rows (floats)."""
    header, rows = read_csv(path)
    if header != DYNAMICS_HEADER:
        raise ValueError("header must be '" + ",".join(DYNAMICS_HEADER) + "'")
    if not rows:
        raise ValueError("no epoch rows")
    if len(rows) > MAX_EPOCHS:
        raise ValueError(f"{len(rows)} epochs logged, but the budget is at most {MAX_EPOCHS}")
    epochs, values = [], []
    for i, r in enumerate(rows, 2):
        if len(r) != 5:
            raise ValueError(f"row {i}: need exactly 5 columns")
        epochs.append(to_int(r[0], f"row {i}: epoch"))
        vals = [to_float(c, f"row {i}: {DYNAMICS_HEADER[k + 1]}") for k, c in enumerate(r[1:])]
        if any(v < 0 or v > 100 for v in vals):
            raise ValueError(f"row {i}: accuracies must be between 0 and 1 (or 0 and 100)")
        values.append(vals)
    if epochs not in (list(range(len(rows))), list(range(1, len(rows) + 1))):
        raise ValueError("epoch must count up by 1 per row, starting at 0 or 1, with no gaps")
    if warnings is not None and len(rows) < MAX_EPOCHS:
        warnings.append(f"{DYNAMICS_FILE}: only {len(rows)} epochs logged "
                        f"(Task 1 asks for the full {MAX_EPOCHS}-epoch budget)")
    return values


def check_scores(path, train_idx, warnings=None):
    """index,score for every training index exactly once. Returns {index: score}."""
    header, rows = read_csv(path)
    if header != ["index", "score"]:
        raise ValueError("header must be 'index,score'")
    scores = {}
    for i, r in enumerate(rows, 2):
        if len(r) != 2:
            raise ValueError(f"row {i}: need exactly 2 columns")
        idx = to_int(r[0], f"row {i}: index")
        if idx not in train_idx:
            raise ValueError(f"row {i}: index {idx} is not in train_idx.txt")
        if idx in scores:
            raise ValueError(f"row {i}: index {idx} appears twice")
        scores[idx] = to_float(r[1], f"row {i}: score")
    missing = train_idx - set(scores)
    if missing:
        raise ValueError(f"{len(missing)} training indices missing (e.g. {sorted(missing)[:3]}); "
                         f"need all {len(train_idx)}")
    if warnings is not None and len(set(scores.values())) == 1:
        warnings.append(f"{SCORES_FILE}: all scores are identical, so the ranking carries no information")
    return scores


# ---------------------------------------------------------------- results.json / report.md
def _get(d, *keys):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return None
        d = d[k]
    return d


def check_results(results, t4_files, issues):
    """Appends human-readable problems to `issues`."""
    if not isinstance(results, dict):
        issues.append("results.json must contain a JSON object")
        return
    for key in ["name", "roll_no"]:
        if not isinstance(results.get(key), str) or not results.get(key).strip():
            issues.append(f"results.json: '{key}' is not filled in")
    if not isinstance(results.get("seed"), int) or isinstance(results.get("seed"), bool):
        issues.append("results.json: 'seed' must be an integer")

    setup = results.get("training_setup")
    if not isinstance(setup, dict):
        issues.append("results.json: 'training_setup' is missing")
    else:
        for key in ["optimizer", "learning_rate_schedule", "augmentation"]:
            if not isinstance(setup.get(key), str) or not setup.get(key).strip():
                issues.append(f"results.json: training_setup.{key} is not filled in")
        for key in ["batch_size", "epochs"]:
            v = setup.get(key)
            if not isinstance(v, int) or isinstance(v, bool) or v <= 0:
                issues.append(f"results.json: training_setup.{key} must be a positive integer")
        if isinstance(setup.get("epochs"), int) and setup["epochs"] > MAX_EPOCHS:
            issues.append(f"results.json: training_setup.epochs is {setup['epochs']}, budget is {MAX_EPOCHS}")

    numeric_fields = [
        ("task1", "noise_rate"),
        ("task1", "val_acc_noisy_labels__clean_ref_model"),
        ("task1", "val_acc_noisy_labels__noisy_ce_model"),
        ("task2", "memorization_start_epoch"),
        ("task2", "early_stop_epoch"),
        ("task2", "val_acc_noisy_labels_at_early_stop"),
        ("task5", "val_acc_synthetic_labels__baseline"),
        ("task5", "val_acc_synthetic_labels__best_method"),
    ]
    for path in numeric_fields:
        v = _get(results, *path)
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            issues.append(f"results.json: {'.'.join(path)} is not filled in (needs a number)")
    nr = _get(results, "task1", "noise_rate")
    if isinstance(nr, (int, float)) and not isinstance(nr, bool) and not 0 <= nr <= 100:
        issues.append("results.json: task1.noise_rate looks out of range")

    if not isinstance(_get(results, "task3", "method"), str) or not _get(results, "task3", "method").strip():
        issues.append("results.json: task3.method is not filled in")

    methods = _get(results, "task4", "methods")
    if not isinstance(methods, list) or not (MIN_T4 <= len(methods) <= MAX_T4):
        issues.append(f"results.json: task4.methods must list {MIN_T4} to {MAX_T4} methods")
        methods = []
    names, files = [], []
    for k, m in enumerate(methods):
        if not isinstance(m, dict):
            issues.append(f"results.json: task4.methods[{k}] must be an object")
            continue
        name, fname, acc = m.get("name"), m.get("file"), m.get("val_acc_noisy_labels")
        if not isinstance(name, str) or not name.strip():
            issues.append(f"results.json: task4.methods[{k}].name is not filled in")
        else:
            names.append(name)
        if not isinstance(fname, str) or fname not in t4_files:
            issues.append(f"results.json: task4.methods[{k}].file '{fname}' is not a t4_*.csv in predictions/")
        else:
            files.append(fname)
        if isinstance(acc, bool) or not isinstance(acc, (int, float)):
            issues.append(f"results.json: task4.methods[{k}].val_acc_noisy_labels is not filled in")
    if len(set(files)) != len(files):
        issues.append("results.json: two task4 methods point to the same file")
    for f in t4_files:
        if f not in files:
            issues.append(f"predictions/{f} is not listed in results.json task4.methods")
    best = _get(results, "task4", "best_method")
    if not isinstance(best, str) or not best.strip():
        issues.append("results.json: task4.best_method is not filled in")
    elif names and best not in names:
        issues.append(f"results.json: task4.best_method '{best}' does not match any method name in task4.methods")


def check_report(path, issues, warnings):
    with open(path, encoding="utf-8") as f:
        text = f.read()
    marks = list(re.finditer(r"(?im)^[\s#>*_\-]*\**\s*Q([1-5])\b", text))
    found = {int(m.group(1)) for m in marks}
    for q in range(1, 6):
        if q not in found:
            issues.append(f"report.md: no answer found for Q{q} (start each answer on a line beginning 'Q{q}')")
    for k, m in enumerate(marks):
        end = marks[k + 1].start() if k + 1 < len(marks) else len(text)
        body = re.sub(r"^[\s#>*_\-]*\**\s*Q\d\b[.:)\]*\s-]*", "", text[m.start():end].strip())
        words = len(body.split())
        if words > 100:
            warnings.append(f"report.md: Q{m.group(1)} is about {words} words (limit is 100)")


# ---------------------------------------------------------------- whole submission
def check_submission(root, train_idx):
    pred_dir = os.path.join(root, "predictions")
    present = set(os.listdir(pred_dir)) if os.path.isdir(pred_dir) else set()
    status, general, warnings = {}, [], []

    def run(name, fn):
        if name not in present:
            status[name] = "MISSING"
            return
        try:
            fn(os.path.join(pred_dir, name))
            status[name] = "OK"
        except Exception as e:  # noqa: BLE001
            status[name] = f"INVALID: {e}"

    for name in PRED_FILES:
        run(name, lambda p: check_pred_file(p, warnings))
    run(DYNAMICS_FILE, lambda p: check_dynamics(p, warnings))
    run(SCORES_FILE, lambda p: check_scores(p, train_idx, warnings))

    t4_files = sorted(f for f in present if T4_PATTERN.match(f))
    for name in t4_files:
        run(name, lambda p: check_pred_file(p, warnings))
    t4_ok = [f for f in t4_files if status[f] == "OK"]
    if not t4_files:
        general.append("no t4_<method>.csv found in predictions/ (need 1 to 3)")
    elif len(t4_files) > MAX_T4:
        general.append(f"{len(t4_files)} t4_*.csv files found, but at most {MAX_T4} are allowed")

    completion = {}
    for t, fs in TASKS.items():
        if fs is None:
            completion[t] = 1.0 if (MIN_T4 <= len(t4_files) <= MAX_T4 and len(t4_ok) == len(t4_files)) else 0.0
        else:
            completion[t] = sum(status.get(f) == "OK" for f in fs) / len(fs)

    known = set(PRED_FILES) | {DYNAMICS_FILE, SCORES_FILE} | set(t4_files)
    extra = sorted(present - known)
    if extra:
        warnings.append(f"Unexpected files in predictions/ (ignored): {', '.join(extra)}")

    rj = os.path.join(root, "results.json")
    if not os.path.isfile(rj):
        general.append("results.json is MISSING")
    else:
        try:
            with open(rj, encoding="utf-8") as f:
                results = json.load(f)
            check_results(results, t4_files, general)
        except Exception as e:  # noqa: BLE001
            general.append(f"results.json is not valid JSON: {e}")

    rp = os.path.join(root, "report.md")
    if not os.path.isfile(rp):
        general.append("report.md is MISSING")
    else:
        check_report(rp, general, warnings)
    if not os.path.isfile(os.path.join(root, "code", "README.md")):
        general.append("code/README.md is MISSING (must say how to reproduce each task)")

    return {"status": status, "t4_files": t4_files, "completion": completion,
            "general": general, "warnings": warnings}


def main():
    if len(sys.argv) != 2:
        print(__doc__)
        sys.exit(1)
    target = sys.argv[1]
    root, tmp = locate_root(target)
    try:
        if root is None:
            print("ERROR: could not find a 'predictions/' folder in your submission.")
            sys.exit(1)
        res = check_submission(root, load_train_idx())
        warnings = list(res["warnings"])
        if os.path.isfile(target) and not re.fullmatch(r".+_cv_bootcamp\.zip", os.path.basename(target)):
            warnings.append("zip name should be <rollno>_cv_bootcamp.zip")

        names = PRED_FILES[:2] + [DYNAMICS_FILE, "t2_early_stop.csv", SCORES_FILE] + res["t4_files"] \
            + PRED_FILES[3:]
        print("\nFile check")
        print("-" * 60)
        for name in names:
            print(f"  {name:26s} {res['status'][name]}")
        if not res["t4_files"]:
            print(f"  {'t4_<method>.csv':26s} MISSING")

        print("\nTask completeness (1.0 = all files valid)")
        print("-" * 60)
        for t, v in res["completion"].items():
            print(f"  {t}: {v:.2f}")

        if res["general"]:
            print("\nOther issues")
            print("-" * 60)
            for m in res["general"]:
                print("  - " + m)
        if warnings:
            print("\nWarnings (do not block submission)")
            print("-" * 60)
            for m in warnings:
                print("  - " + m)

        ok = all(v == 1.0 for v in res["completion"].values()) and not res["general"]
        print("\n" + ("All good." if ok else "Fix the issues above (or submit what you have)."))
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)


if __name__ == "__main__":
    main()

## Training (each step is skipped if its output already exists on Drive)

In [ ]:
import os, subprocess
CODE = f"{ROOT}/submission/code"
WORK = f"{ROOT}/work"

def run(done_file, *args):
    if os.path.exists(f"{WORK}/{done_file}"):
        print("skip (already done):", " ".join(args))
        return
    cmd = ["python", "-u", "nl.py", *args]
    print(">>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, "run failed (see output above)"

### Task 1 (+ Task 2 dynamics via `--track`)

In [ ]:
run("t1_noisy/test_logits.npy", "train", "--tag", "t1_noisy", "--labels", "noisy", "--method", "ce", "--track")

In [ ]:
run("t1_clean/test_logits.npy", "train", "--tag", "t1_clean", "--labels", "clean", "--method", "ce")

### Task 3: 4-fold cross-validated confidence

In [ ]:
run("cv_noisy/heldout_probs.npy", "cv", "--labels", "noisy")

### Task 4: robust methods

In [ ]:
run("t4_elr/test_logits.npy", "train", "--tag", "t4_elr", "--labels", "noisy", "--method", "elr")

In [ ]:
run("t4_relabel/test_logits.npy", "train", "--tag", "t4_relabel", "--labels", "noisy", "--method", "relabel", "--ce-tag", "t1_noisy")

In [ ]:
run("t4_gce/test_logits.npy", "train", "--tag", "t4_gce", "--labels", "noisy", "--method", "gce")

### Task 5: synthetic (symmetric) noise: CE baseline and the best Task 4 method

In [ ]:
run("t5_ce/test_logits.npy", "train", "--tag", "t5_ce", "--labels", "synthetic", "--method", "ce", "--track")

In [ ]:
import sys
sys.path.insert(0, CODE); os.chdir(CODE)
import summarize
BEST = summarize.best_t4_method()
print("best Task 4 method:", BEST)
if BEST == "relabel":
    run("cv_synthetic/heldout_probs.npy", "cv", "--labels", "synthetic")
    run("t5_best/test_logits.npy", "train", "--tag", "t5_best", "--labels", "synthetic", "--method", "relabel",
        "--ce-tag", "t5_ce")
else:
    run("t5_best/test_logits.npy", "train", "--tag", "t5_best", "--labels", "synthetic", "--method", BEST)

## Predictions, results.json, analysis

In [ ]:
import json
SUB = f"{ROOT}/submission"
rj = f"{SUB}/results.json"
r = json.load(open(rj)) if os.path.exists(rj) else {}
r["name"], r["roll_no"] = NAME, ROLL_NO
json.dump(r, open(rj, "w"), indent=2)
!cd "{CODE}" && python nl.py summarize

## Report

The text below is the report from the reference run. **Your numbers will differ slightly.** Update them from
`submission/code/analysis.md` (printed below) before you submit, and keep each answer at 100 words or fewer.

In [ ]:
print(open(f"{ROOT}/submission/code/analysis.md").read())

In [ ]:
%%writefile /content/drive/MyDrive/cv_bootcamp/submission/report.md
# CV Bootcamp report: learning from noisy labels (CIFAR-10N "worst")

Supporting tables (confusion matrix, per-epoch dynamics, all metrics) are in `code/analysis.md`.

## Task 1 confusion matrix

Noisy vs clean labels on the 45,000-image training split (rows = clean label, columns = noisy label). Noise rate: 40.2%.

| clean \ noisy | airplane | automobile | bird | cat | deer | dog | frog | horse | ship | truck |
|---|---|---|---|---|---|---|---|---|---|---|
| airplane | 2925 | 293 | 314 | 89 | 94 | 105 | 85 | 75 | 379 | 146 |
| automobile | 191 | 2657 | 63 | 116 | 80 | 68 | 60 | 64 | 71 | 1126 |
| bird | 263 | 165 | 2842 | 212 | 254 | 216 | 222 | 147 | 106 | 77 |
| cat | 180 | 160 | 354 | 2168 | 194 | 924 | 206 | 106 | 91 | 86 |
| deer | 184 | 142 | 316 | 184 | 2195 | 491 | 174 | 683 | 94 | 50 |
| dog | 150 | 131 | 240 | 834 | 183 | 2620 | 123 | 136 | 55 | 61 |
| frog | 170 | 166 | 373 | 350 | 229 | 257 | 2676 | 124 | 96 | 62 |
| horse | 179 | 120 | 111 | 100 | 233 | 289 | 101 | 3246 | 77 | 50 |
| ship | 369 | 333 | 94 | 65 | 96 | 79 | 71 | 99 | 3076 | 214 |
| truck | 180 | 1247 | 81 | 87 | 78 | 81 | 52 | 68 | 115 | 2486 |

Q1. Which class pairs account for most of the noise?

The noise rate is 40.2% on the 45k training split. The noise is concentrated in visually similar pairs: automobile↔truck accounts for 13.1% of all wrong labels (truck→automobile 1,247, automobile→truck 1,126), cat↔dog for 9.7%, deer↔horse for 5.1% and airplane↔ship for 4.1%. Deer→dog (491) and bird→frog/deer add more. These pairs are strongly asymmetric across classes: cat and deer images are mislabeled 51% of the time, horse only 28%. So the noise is class-dependent and systematic, not uniform.

Q2. At which epoch does memorization of wrong labels start, and what is the evidence?

Epoch 32. From epoch 4 to 30, accuracy on the mislabeled subset measured against the wrong label stays flat at 12–15%, and 60–66% of those images are predicted as their true class. From epoch 32 the wrong-label fit rises steadily to 69% at epoch 60, while true-class predictions fall to 21%. Clean validation accuracy peaks at 79.4% (epoch 41) and drops to 69.3%. Noisy validation accuracy peaks at epoch 25 (55.6%), which we use for early stopping (clean val 78.7%), and decays to 51.0%.

Q3. Which noise-score signal worked best? Describe one failure mode.

Per-sample loss averaged over all 60 epochs of the noisy CE run was best (AUROC 0.958, precision@k 0.862), slightly ahead of 4-fold cross-validated confidence (0.942 / 0.837). The submitted score averages both ranks (0.956 / 0.856). Final-epoch loss alone is poor (0.737) because wrong labels are memorized by then. Failure mode: systematic confusions are missed. Only 64% of mislabeled automobile↔truck and 60% of cat↔dog images are flagged, against about 90% for deer↔dog or bird↔frog. Annotators mislabel ambiguous images consistently, so the model learns to agree with them.

Q4. Why did your best Task 4 method help, or not help?

ELR was best: 58.3% noisy validation accuracy, against 55.6% for early-stopped CE (clean validation 86.7% vs 78.7%, measured only). Early in training the network fits mostly correct labels. ELR keeps a running average of each image's predictions and penalises moving away from it, cancelling a wrong label's gradient once the model is confident in the true class. Memorization never sets in: its best epoch is the last. Filter+relabel also helped (57.4%), but it dropped 9.4k images and inherits Task 3's errors on automobile/truck and cat/dog.

Q5. Did the ranking of methods change between real and synthetic noise? Why?

No. ELR beats CE on both: 54.0% vs 50.4% synthetic validation accuracy, against 58.3% vs 55.6% on real noise. Clean validation accuracy (measured only) is 90.6% vs 83.7% on synthetic and 86.7% vs 78.7% on real. At the same 40% rate, synthetic noise is easier: CE starts memorizing later (epoch 42 vs 32) and fits fewer wrong labels (48% vs 69%). Random flips are inconsistent across similar images, so their gradients cancel. Human errors repeat on the same ambiguous images (automobile/truck, cat/dog), so they look like learnable signal.

## Zip, format check and download

In [ ]:
import zipfile, pathlib
roll = ROLL_NO.lower()
zpath = pathlib.Path(ROOT) / f"{roll}_cv_bootcamp.zip"
src = pathlib.Path(SUB)
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(src.rglob("*")):
        rel = p.relative_to(src)
        if p.is_file() and rel.parts[0] in ("code", "predictions", "results.json", "report.md") \
                and "__pycache__" not in rel.parts and p.suffix != ".ipynb":
            z.write(p, pathlib.Path(f"{roll}_cv_bootcamp") / rel)
print(zpath, zpath.stat().st_size // 1024, "KB")
!cd "{ROOT}" && python check_format.py "{zpath}"

In [ ]:
from google.colab import files
files.download(str(zpath))